# 21 — Technique: Strategy A (reverse transliteration)

**Source.** The Swa-Bhasha lineage in `research/romanized_sinhala/` — Sumanathilaka (2022) →
IndoNLP-2025 BERT reverse-transliterator (Perera et al.) → Swa-Bhasha Resource Hub (2026,
arXiv:2507.09245). Reported **WER 0.085-0.09, BLEU-4 ~0.80**; the fine-tuned Swa-Bhasha-mBART wins
their code-mixed benchmark at BLEU 49.70, beating few-shot Gemini's 33.43.

`model-research.md` §5 lists three ways to handle romanized input, "all competing as equals":

- **A** — reverse-transliterate romanized → native script, then run the native-script model.
- **B** — process romanized directly (the current pipeline).
- **C** — augmentation at fine-tuning time (notebook 22).

A has never been tested here. This notebook tests it.

---

## Read this before quoting any number below

**Our Singlish is rule-generated from Sinhala.** `singlishify.py` maps Sinhala tokens to romanized
forms through a deterministic table (`singlish_overrides.py`: `කාඩ්` → `card`, always). Reverse-
transliterating it is therefore **inverting a function we ourselves applied**, and it will recover
the source text at an error rate no human-typed romanized input could ever match.

[`08_word_tokenizer_comparison.ipynb`](08_word_tokenizer_comparison.ipynb) §6 puts a number on
this: **Singlish dev OOV is 1.18%, identical to English's**, because the orthography is perfectly
consistent by construction. Real romanized text has no standard spelling at all.

So Strategy A will very likely post an excellent score here, and that score will be close to
meaningless. It is run because the technique was requested and because the *mechanics* are worth
having in place for when human-typed data exists — not because the result will settle A vs B.

**Tamilish is marginally less circular** (it came from the translation pass rather than a rule),
which is why it is reported separately rather than pooled.

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

REPO = Path.cwd()
while not (REPO / "ml" / "swiftbench").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "ml"))

import numpy as np, pandas as pd
import swiftbench as sb
from swiftbench import config, data, imbalance, metrics, models, splits, tokenize as sbtok

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

AUTHOR = "sithija"
LANGS = config.LANGUAGES
POS = config.SENTIMENT_POSITIVE_CLASS
print("split sha:", splits.sha())

In [ ]:
def boot_ci(y_true, y_pred, ids, n_boot=1000, seed=42):
    # 95% CI for negative_f1, resampled over ticket id.
    yt = (np.asarray(y_true) == POS); yp = (np.asarray(y_pred) == POS)
    groups = [g.to_numpy() for _, g in pd.Series(np.arange(len(yt))).groupby(np.asarray(ids))]
    rng = np.random.default_rng(seed); n = len(groups); out = np.empty(n_boot)
    for i in range(n_boot):
        idx = np.concatenate([groups[j] for j in rng.integers(0, n, n)])
        tp = (yt[idx] & yp[idx]).sum(); fp = (~yt[idx] & yp[idx]).sum(); fn = (yt[idx] & ~yp[idx]).sum()
        out[i] = 0.0 if tp == 0 else 2 * tp / (2 * tp + fp + fn)
    return tuple(np.percentile(out, [2.5, 97.5]))


def fit_champion(df, task="sentiment"):
    col = data.label_column(task)
    fit = imbalance.resample(df, col, "class_weight")
    clf = models.build("tfidf-svm", class_weight="balanced", C=0.5 if task == "sentiment" else 1.0)
    clf.fit(fit[config.TEXT_COLUMN], fit[col])
    return clf

## 1. The transliterator

`deshanksuman/swabhashambart50SinhalaTransliteration` — the mBART-50 checkpoint from the Swa-Bhasha
Resource Hub. Generation over several thousand rows is the slow part of this notebook, so it is
cached to disk on first run.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

TRANSLITERATOR = "deshanksuman/swabhashambart50SinhalaTransliteration"
CACHE = REPO / "ml" / "reports" / "strategy_a_transliterated.csv"
DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
SMOKE = True          # <- set False for the full dev set

tok = AutoTokenizer.from_pretrained(TRANSLITERATOR)
mt = AutoModelForSeq2SeqLM.from_pretrained(TRANSLITERATOR).to(DEVICE).eval()
print("loaded", TRANSLITERATOR, "on", DEVICE)

In [ ]:
def transliterate(texts, batch_size=32, max_new_tokens=96):
    out = []
    for i in range(0, len(texts), batch_size):
        chunk = texts[i : i + batch_size]
        enc = tok(chunk, return_tensors="pt", padding=True, truncation=True, max_length=128).to(DEVICE)
        with torch.no_grad():
            gen = mt.generate(**enc, max_new_tokens=max_new_tokens, num_beams=1)
        out.extend(tok.batch_decode(gen, skip_special_tokens=True))
        if i % (batch_size * 10) == 0:
            print(f"    {i}/{len(texts)}", flush=True)
    return out


dev = splits.get(LANGS, "dev")
romanized = dev[dev.language.isin(["singlish", "tamilish"])].copy()
if SMOKE:
    romanized = romanized.groupby("language", group_keys=False).head(60)

if CACHE.exists() and not SMOKE:
    cached = pd.read_csv(CACHE)
    romanized = romanized.merge(cached[["id", "language", "transliterated"]], on=["id", "language"], how="left")
else:
    for lang in romanized.language.unique():
        m = romanized.language == lang
        print(f"  transliterating {lang} ({m.sum()} rows)")
        romanized.loc[m, "transliterated"] = transliterate(romanized.loc[m, config.TEXT_COLUMN].tolist())
    if not SMOKE:
        romanized[["id", "language", "transliterated"]].to_csv(CACHE, index=False)

display(romanized[[config.TEXT_COLUMN, "transliterated", "language"]].head(8))

## 2. Did it actually produce Sinhala?

Before scoring anything: check the output is native script and not an echo of the input. A
transliterator that passes text through unchanged would silently reduce Strategy A to Strategy B.

In [ ]:
romanized["out_script"] = romanized.transliterated.fillna("").map(sbtok.script_of)
display(pd.crosstab(romanized.language, romanized.out_script))

unchanged = (romanized.transliterated.fillna("").str.strip() == romanized[config.TEXT_COLUMN].str.strip()).mean()
print(f"output identical to input: {unchanged:.1%}")
print("If this is high, the transliterator is not doing its job and section 3 is meaningless.")

## 3. Strategy A vs Strategy B

Same evaluation rows, two pipelines:

- **B (incumbent)** — the multilingual classical model reads the romanized text directly.
- **A** — the transliterated text is scored by a model trained on **native Sinhala only**.

Reported per language, never pooled, because the circularity differs between them.

In [ ]:
train = splits.get(LANGS, "train")
model_b = fit_champion(train)                                   # multilingual, reads romanized
model_a = fit_champion(splits.get(["sinhala"], "train"))        # native Sinhala only

rows = []
for lang in romanized.language.unique():
    sub = romanized[romanized.language == lang].dropna(subset=["transliterated"])
    yt = sub.sentiment.to_numpy()
    for name, texts, mdl in [("B: direct romanized", sub[config.TEXT_COLUMN], model_b),
                             ("A: transliterated -> sinhala model", sub.transliterated, mdl_a := model_a)]:
        pred = mdl.predict(texts)
        sc = metrics.score(yt, pred, "sentiment")
        lo, hi = boot_ci(yt, pred, sub.id.to_numpy(), n_boot=400)
        rows.append({"language": lang, "strategy": name, "negative_f1": sc["negative_f1"],
                     "precision": sc["negative_precision"], "recall": sc["negative_recall"],
                     "accuracy": sc["accuracy"], "ci_low": lo, "ci_high": hi, "n": len(sub)})

comparison = pd.DataFrame(rows)
display(comparison)
comparison.to_csv(REPO / "ml" / "reports" / "technique_strategy_a.csv", index=False)

## 4. Verdict

Whatever the numbers say, they carry the caveat from the top of this notebook, and the write-up
must carry it too.

- If **A wins on Singlish**: expected, and largely an artifact of inverting `singlishify.py`.
  Not evidence for deploying A.
- If **A wins on Tamilish** by a similar margin: more interesting, since Tamilish was not produced
  by a reversible rule — but it is still machine-generated.
- If **A loses even here**: that *is* informative. Strategy A gets its best possible case in this
  setup, so losing under favourable conditions is real evidence against it.

The honest test needs human-typed romanized tickets. Record that as the blocking dependency rather
than reporting an A-vs-B winner.